# 09 · Anomaly Detection via Prediction Intervals

TimesFM has no built-in anomaly detector, but its **quantile bands** are a
natural one: any actual value outside the 80% (or 90%) interval is statistically
unusual. This is the engine behind a "smart alerting" product.

In [ ]:
import torch
import numpy as np
import timesfm

torch.set_float32_matmul_precision("high")

# Downloads ~800 MB of weights the first time, then caches in ~/.cache/huggingface/
model = timesfm.TimesFM_2p5_200M_torch.from_pretrained(
    "google/timesfm-2.5-200m-pytorch"
)

model.compile(
    timesfm.ForecastConfig(
        max_context=1024,
        max_horizon=256,
        normalize_inputs=True,
        use_continuous_quantile_head=True,
        force_flip_invariance=True,
        infer_is_positive=True,
        fix_quantile_crossing=True,
    )
)
print("Model loaded and compiled.")

In [ ]:
# A clean series with a couple of injected anomalies near the end
rng = np.random.default_rng(99)
t = np.arange(300)
clean = (100 + 15*np.sin(2*np.pi*t/24) + rng.normal(0, 3, t.size)).astype(np.float32)
series = clean.copy()
series[270] += 45   # spike
series[288] -= 40   # dip

## Strategy: walk-forward one-step-ahead

For each recent point, forecast it from its past and check whether the true
value falls inside the predicted band.

In [ ]:
IDX_Q10, IDX_Q90 = 1, 9
start = 250          # begin checking here
records = []
for i in range(start, len(series)):
    ctx = series[:i]                       # everything before point i
    p, q = model.forecast(horizon=1, inputs=[ctx])
    lo, hi = q[0, 0, IDX_Q10], q[0, 0, IDX_Q90]
    actual = series[i]
    is_anom = actual < lo or actual > hi
    records.append((i, actual, float(lo), float(hi), bool(is_anom)))

anoms = [r for r in records if r[4]]
print(f"checked {len(records)} points, flagged {len(anoms)} anomalies:")
for i, a, lo, hi, _ in anoms:
    print(f"  t={i}: actual={a:.1f} outside [{lo:.1f}, {hi:.1f}]")

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

idx = [r[0] for r in records]
act = [r[1] for r in records]
lo  = [r[2] for r in records]
hi  = [r[3] for r in records]
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(range(len(series)), series, color="tab:blue", lw=1, label="series")
ax.fill_between(idx, lo, hi, color="tab:green", alpha=0.2, label="expected 80% band")
for i, a, *_ , flag in records:
    if flag:
        ax.scatter(i, a, color="red", zorder=5, s=60)
ax.set_title("Anomaly detection (red = outside expected band)")
ax.legend(); fig.tight_layout(); fig.savefig("anomalies.png", dpi=130)
print("saved anomalies.png")

## Severity levels

| Severity | Condition | Meaning |
| -------- | --------- | ------- |
| Normal   | inside 80% band | expected |
| Warning  | outside 80% band | unusual but possible |
| Critical | outside 90% band | statistically rare (<10%) |

> See `../timesfm-forecasting/examples/anomaly-detection/` for a fuller example
> that also de-trends and adds a z-score.